# CNN Architecture Experiments (Seat Cover)

Notebook-style training workflow with **3 CNN models in separate cells**.


In [3]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)


Using device: mps


## Config + Transforms (same as notebook)


In [25]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
LR = 0.0003
criterion = nn.CrossEntropyLoss()

# Safety toggle so this notebook can be executed without starting long training runs
RUN_FULL_TRAINING = True

class PadResize:
    def __init__(self, size):
        self.size = size

    def __call__(self, img):
        w, h = img.size
        max_side = max(w, h)

        pad_w = (max_side - w) // 2
        pad_h = (max_side - h) // 2

        padding = (pad_w, pad_h, max_side - w - pad_w, max_side - h - pad_h)
        img = transforms.functional.pad(img, padding, fill=0)
        img = img.resize((self.size, self.size))
        return img

train_transform_aug = transforms.Compose([
    PadResize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

train_transform_noaug = transforms.Compose([
    PadResize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

val_test_transform = transforms.Compose([
    PadResize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

print("Transforms created")
print("RUN_FULL_TRAINING:", RUN_FULL_TRAINING)


Transforms created
RUN_FULL_TRAINING: True


## Seat Cover Datasets + Dataloaders


In [17]:
seat_train_dir = "data/splits/Seat Cover/train"
seat_val_dir   = "data/splits/Seat Cover/val"
seat_test_dir  = "data/splits/Seat Cover/test"

for p in [seat_train_dir, seat_val_dir, seat_test_dir]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing dataset path: {p}")

seat_train_noaug = datasets.ImageFolder(seat_train_dir, transform=train_transform_noaug)
seat_train_aug   = datasets.ImageFolder(seat_train_dir, transform=train_transform_aug)
seat_val         = datasets.ImageFolder(seat_val_dir, transform=val_test_transform)
seat_test        = datasets.ImageFolder(seat_test_dir, transform=val_test_transform)

seat_train_loader_noaug = DataLoader(seat_train_noaug, batch_size=32, shuffle=True)
seat_train_loader_aug   = DataLoader(seat_train_aug, batch_size=32, shuffle=True)
seat_val_loader         = DataLoader(seat_val, batch_size=32, shuffle=False)
seat_test_loader        = DataLoader(seat_test, batch_size=32, shuffle=False)

print("Seat datasets loaded")
print("Classes:", seat_train_noaug.classes)
print("Train/Val/Test sizes:", len(seat_train_noaug), len(seat_val), len(seat_test))


Seat datasets loaded
Classes: ['Elongated', 'Round']
Train/Val/Test sizes: 117 24 27


## Shared CNN Block


In [18]:
class ConvBNReLUPoolDrop(nn.Module):
    def __init__(self, in_ch, out_ch, p_drop):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(p_drop),
        )

    def forward(self, x):
        return self.block(x)


## Model 1: BaselineCNN


In [19]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            ConvBNReLUPoolDrop(3, 32, 0.10),
            ConvBNReLUPoolDrop(32, 64, 0.15),
            ConvBNReLUPoolDrop(64, 128, 0.20),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


## Model 2: BNDropoutCNN


In [20]:
class BNDropoutCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            ConvBNReLUPoolDrop(3, 32, 0.15),
            ConvBNReLUPoolDrop(32, 64, 0.20),
            ConvBNReLUPoolDrop(64, 128, 0.25),
            ConvBNReLUPoolDrop(128, 128, 0.30),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


## Model 3: DeeperCNN


In [21]:
class DeeperCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            ConvBNReLUPoolDrop(3, 32, 0.10),
            ConvBNReLUPoolDrop(32, 64, 0.15),
            ConvBNReLUPoolDrop(64, 128, 0.20),
            ConvBNReLUPoolDrop(128, 256, 0.25),
            ConvBNReLUPoolDrop(256, 256, 0.30),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.50),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


## Training Function


In [22]:
def train_model(model, train_loader, val_loader, optimizer, name):
    train_losses, train_accs, val_accs = [], [], []
    best_val_acc, best_state = 0.0, None

    for epoch in range(EPOCHS):
        model.train()
        running_loss, correct_train, total_train = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct_train / total_train

        model.eval()
        correct_val, total_val = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                preds = outputs.argmax(dim=1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)

        val_acc = correct_val / total_val

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"{name} | Epoch [{epoch+1}/{EPOCHS}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return {
        "train_losses": train_losses,
        "train_accs": train_accs,
        "val_accs": val_accs,
        "best_val_acc": best_val_acc,
    }


## Sanity Check (runs fast)


In [23]:
images, labels = next(iter(seat_train_loader_noaug))
print("Batch shape:", images.shape, labels.shape)

for model_name, model in [
    ("BaselineCNN", BaselineCNN(2)),
    ("BNDropoutCNN", BNDropoutCNN(2)),
    ("DeeperCNN", DeeperCNN(2)),
]:
    logits = model(images[:2])
    print(model_name, "->", tuple(logits.shape))

results_summary = {}


Batch shape: torch.Size([32, 3, 224, 224]) torch.Size([32])
BaselineCNN -> (2, 2)
BNDropoutCNN -> (2, 2)
DeeperCNN -> (2, 2)


## Train BaselineCNN (separate cell)


In [26]:
if RUN_FULL_TRAINING:
    baseline_model = BaselineCNN(num_classes=2).to(device)
    baseline_optimizer = optim.Adam(baseline_model.parameters(), lr=LR, weight_decay=1e-4)
    baseline_history = train_model(baseline_model, seat_train_loader_noaug, seat_val_loader, baseline_optimizer, "BaselineCNN")
    results_summary["BaselineCNN"] = baseline_history["best_val_acc"]
    print(f"BaselineCNN best validation accuracy: {baseline_history['best_val_acc']:.4f}")
else:
    print("RUN_FULL_TRAINING is False: skipping BaselineCNN training")


BaselineCNN | Epoch [1/50] Train Loss: 0.7013 | Train Acc: 0.4872 | Val Acc: 0.4167
BaselineCNN | Epoch [2/50] Train Loss: 0.6822 | Train Acc: 0.5470 | Val Acc: 0.4167
BaselineCNN | Epoch [3/50] Train Loss: 0.6965 | Train Acc: 0.5470 | Val Acc: 0.5833
BaselineCNN | Epoch [4/50] Train Loss: 0.6887 | Train Acc: 0.5043 | Val Acc: 0.5833
BaselineCNN | Epoch [5/50] Train Loss: 0.7022 | Train Acc: 0.5128 | Val Acc: 0.5833
BaselineCNN | Epoch [6/50] Train Loss: 0.6787 | Train Acc: 0.5812 | Val Acc: 0.5833
BaselineCNN | Epoch [7/50] Train Loss: 0.6841 | Train Acc: 0.5385 | Val Acc: 0.5833
BaselineCNN | Epoch [8/50] Train Loss: 0.6909 | Train Acc: 0.5128 | Val Acc: 0.5833
BaselineCNN | Epoch [9/50] Train Loss: 0.6833 | Train Acc: 0.5556 | Val Acc: 0.5833
BaselineCNN | Epoch [10/50] Train Loss: 0.6829 | Train Acc: 0.5641 | Val Acc: 0.5833
BaselineCNN | Epoch [11/50] Train Loss: 0.6689 | Train Acc: 0.5641 | Val Acc: 0.5417
BaselineCNN | Epoch [12/50] Train Loss: 0.6796 | Train Acc: 0.5556 | Val A

## Train BNDropoutCNN (separate cell)


In [27]:
if RUN_FULL_TRAINING:
    bndrop_model = BNDropoutCNN(num_classes=2).to(device)
    bndrop_optimizer = optim.Adam(bndrop_model.parameters(), lr=LR, weight_decay=1e-4)
    bndrop_history = train_model(bndrop_model, seat_train_loader_noaug, seat_val_loader, bndrop_optimizer, "BNDropoutCNN")
    results_summary["BNDropoutCNN"] = bndrop_history["best_val_acc"]
    print(f"BNDropoutCNN best validation accuracy: {bndrop_history['best_val_acc']:.4f}")
else:
    print("RUN_FULL_TRAINING is False: skipping BNDropoutCNN training")


BNDropoutCNN | Epoch [1/50] Train Loss: 0.7519 | Train Acc: 0.5299 | Val Acc: 0.5833
BNDropoutCNN | Epoch [2/50] Train Loss: 0.7508 | Train Acc: 0.4872 | Val Acc: 0.5833
BNDropoutCNN | Epoch [3/50] Train Loss: 0.7282 | Train Acc: 0.4615 | Val Acc: 0.5833
BNDropoutCNN | Epoch [4/50] Train Loss: 0.7688 | Train Acc: 0.4444 | Val Acc: 0.5833
BNDropoutCNN | Epoch [5/50] Train Loss: 0.6985 | Train Acc: 0.5128 | Val Acc: 0.5833
BNDropoutCNN | Epoch [6/50] Train Loss: 0.6922 | Train Acc: 0.5214 | Val Acc: 0.5833
BNDropoutCNN | Epoch [7/50] Train Loss: 0.7292 | Train Acc: 0.5214 | Val Acc: 0.5833
BNDropoutCNN | Epoch [8/50] Train Loss: 0.7384 | Train Acc: 0.5299 | Val Acc: 0.5833
BNDropoutCNN | Epoch [9/50] Train Loss: 0.6788 | Train Acc: 0.5983 | Val Acc: 0.5833
BNDropoutCNN | Epoch [10/50] Train Loss: 0.7091 | Train Acc: 0.5385 | Val Acc: 0.5833
BNDropoutCNN | Epoch [11/50] Train Loss: 0.7409 | Train Acc: 0.4786 | Val Acc: 0.5417
BNDropoutCNN | Epoch [12/50] Train Loss: 0.7142 | Train Acc: 0.

## Train DeeperCNN (separate cell)


In [28]:
if RUN_FULL_TRAINING:
    deeper_model = DeeperCNN(num_classes=2).to(device)
    deeper_optimizer = optim.Adam(deeper_model.parameters(), lr=LR, weight_decay=1e-4)
    deeper_history = train_model(deeper_model, seat_train_loader_noaug, seat_val_loader, deeper_optimizer, "DeeperCNN")
    results_summary["DeeperCNN"] = deeper_history["best_val_acc"]
    print(f"DeeperCNN best validation accuracy: {deeper_history['best_val_acc']:.4f}")
else:
    print("RUN_FULL_TRAINING is False: skipping DeeperCNN training")


DeeperCNN | Epoch [1/50] Train Loss: 0.7061 | Train Acc: 0.4701 | Val Acc: 0.5833
DeeperCNN | Epoch [2/50] Train Loss: 0.6930 | Train Acc: 0.5470 | Val Acc: 0.5833
DeeperCNN | Epoch [3/50] Train Loss: 0.6849 | Train Acc: 0.5641 | Val Acc: 0.5833
DeeperCNN | Epoch [4/50] Train Loss: 0.6930 | Train Acc: 0.4701 | Val Acc: 0.5833
DeeperCNN | Epoch [5/50] Train Loss: 0.6931 | Train Acc: 0.5726 | Val Acc: 0.5833
DeeperCNN | Epoch [6/50] Train Loss: 0.6995 | Train Acc: 0.5128 | Val Acc: 0.5833
DeeperCNN | Epoch [7/50] Train Loss: 0.6800 | Train Acc: 0.5812 | Val Acc: 0.5833
DeeperCNN | Epoch [8/50] Train Loss: 0.6969 | Train Acc: 0.5726 | Val Acc: 0.5833
DeeperCNN | Epoch [9/50] Train Loss: 0.6931 | Train Acc: 0.5726 | Val Acc: 0.5833
DeeperCNN | Epoch [10/50] Train Loss: 0.6798 | Train Acc: 0.6154 | Val Acc: 0.5833
DeeperCNN | Epoch [11/50] Train Loss: 0.6898 | Train Acc: 0.5556 | Val Acc: 0.5833
DeeperCNN | Epoch [12/50] Train Loss: 0.6980 | Train Acc: 0.4957 | Val Acc: 0.5833
DeeperCNN | E

## Final Summary


In [15]:
if results_summary:
    print("Final Results (best validation accuracy)")
    for name, acc in results_summary.items():
        print(f"- {name}: {acc:.4f}")
    best_name, best_acc = max(results_summary.items(), key=lambda x: x[1])
    print(f"\nBest architecture: {best_name} ({best_acc:.4f})")
else:
    print("No training results yet. Set RUN_FULL_TRAINING = True and run the three training cells.")


No training results yet. Set RUN_FULL_TRAINING = True and run the three training cells.
